# NISAR L2 GCOV surface-water mapping — walkthrough

InGARSS 2026 YP Hackathon, Phase 1. This notebook **reads the outputs of the experiment runs** and shows the main results;
it does not re-run the heavy processing. Every number here comes from a run folder under `experiments/`, which also holds
the frozen config, provenance (git commit, package versions, input granules), metrics and maps.

**Reproduce from scratch** (≈ 1 h for the bbox experiments, several hours for L10):
```bash
uv sync
uv run nisar-water run experiments/L00_data_inventory/config.yaml          # ASF inventory
uv run nisar-water fetch experiments/L00_data_inventory/runs/<ts>/tables/inventory.csv --track 98   # bbox windows
uv run nisar-water run experiments/R00_reference_build/config.yaml         # S2 / S1 / JRC references, split
for e in L01_preprocessing_separability L02_global_threshold L03_auto_threshold L04_tiled_threshold \
         L05_dualpol_combination L06_contextual_refinement L07_supervised_rf_lgbm L08_polarimetric_features \
         L09_change_detection; do uv run nisar-water run experiments/$e/config.yaml; done
uv run nisar-water download ...   # full asc 98/14 granules, then:
uv run nisar-water run experiments/L10_full_footprint/config.yaml
```

In [ ]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import Image, Markdown, display

from nisar_water import REPO_ROOT
from nisar_water.report import latest_run

EXP = REPO_ROOT / "experiments"
def run(exp): return latest_run(EXP / exp)
print((EXP / "INDEX.md").read_text()[:4000])

## 1. Data inventory (Task A)
L2 GCOV granules intersecting the reference bbox, from ASF.

In [ ]:
r = run("L00_data_inventory")
inv = pd.read_csv(r / "tables" / "inventory.csv")
display(inv.groupby(["direction", "track", "frame", "polarization_code"]).agg(n=("granule", "size"),
        first=("start_time", "min"), last=("start_time", "max"), gb=("size_gb", "mean")).round(2))
for f in sorted((r / "figures").glob("*.png"))[:2]: display(Image(f, width=700))

## 2. Water-mask method ladder (Task B, L-band)
Test-block F1 against same-date Sentinel-2 (5 dates), pooled confusion matrix.

In [ ]:
rows = []
for exp in ["L02_global_threshold", "L03_auto_threshold", "L04_tiled_threshold", "L05_dualpol_combination",
            "L06_contextual_refinement", "L07_supervised_rf_lgbm"]:
    m = json.loads((run(exp) / "metrics.json").read_text())
    rows.append({"experiment": exp, "selected": m.get("best_variant"), "test_F1_pooled": round(m["headline"]["value"], 3)})
display(pd.DataFrame(rows))
r = run("L07_supervised_rf_lgbm")
s = pd.read_csv(r / "tables" / "scores.csv")
display(s.pivot(index="date", columns="variant", values="water_km2").round(1))
display(Image(r / "figures" / "water_lgbm_rel_full_mmu25_2026-01-15.png", width=650))

## 3. Polarimetric assessment (Task H, L08)

In [ ]:
r = run("L08_polarimetric_features")
if r:
    display(Markdown((r / "results.md").read_text()))

## 4. Change detection and new water (Tasks D, E; L09)

In [ ]:
r = run("L09_change_detection")
display(pd.read_csv(r / "tables" / "change_assessment.csv").round(3))
display(pd.read_csv(r / "tables" / "change_areas.csv").query("method == 'pcc'").drop(columns="method").round(2))
for f in ["change_pcc_2026-01-15_2026-09-12.png", "new_water_pcc_2026-01-15_2026-09-12.png"]:
    display(Image(r / "figures" / f, width=650))

## 5. Full scene footprint (L10)

In [ ]:
r = run("L10_full_footprint")
if r:
    display(Markdown((r / "results.md").read_text()))
    for f in sorted((r / "figures").glob("water_full_*.png"))[:3]: display(Image(f, width=650))